In [2]:
import os

In [4]:
import sys
sys.executable

'/Users/juhyeongpang/Desktop/Projects/Quant/quant/bin/python'

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk

import tensorflow as tf
import keras
import keras_nlp
import keras_hub

import yfinance as yf

In [26]:
gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("gemma2_2b_en", dtype="float16")

normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


In [27]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

In [28]:
prompt = template.format(
    instruction = "What is your favorite movie?",
    response = ""
)

In [29]:
gemma_lm.backbone.enable_lora(rank = 4)
# gemma_lm.summary()

In [30]:
# help(gemma_lm.backbone)

In [21]:
sentiment_raw = pd.read_csv(os.path.join("sentiment_dataset", "all-data.csv"), encoding='latin1', header = None)
sentiment_raw.columns = ["Output", "Input"]
sentiment_dataset = sentiment_raw[["Input", "Output"]]

sentiment_dataset["Output"].value_counts()

Output
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64

In [33]:
def create_prompt(row):
    return f"Instruction:\n{row['Instruction']}\n\nInput:\n{row['Input']}\n\nResponse:\n{row['Output']}"

sentiment_dataset['text'] = sentiment_dataset.apply(create_prompt, axis=1)

sentiment_dataset_combined = sentiment_dataset['text'].values

In [34]:
gemma_lm.preprocessor.sequence_length = 256
optimizer = keras.optimizers.AdamW(
    learning_rate = 5e-5,
    weight_decay = 0.01
)

optimizer.exclude_from_weight_decay(var_names = ["bias", "scale"])

gemma_lm.compile(
    loss = keras.losses.SparseCategoricalCrossentropy(from_logits = True),
    optimizer = optimizer,
    weighted_metrics = [keras.metrics.SparseCategoricalAccuracy()]
)

In [53]:
gemma_lm.fit(sentiment_dataset_combined[:200], epochs = 1, batch_size = 1)

200/200 ━━━━━━━━━━━━━━━━━━━━ 7560s 38s/step - loss: 0.7771 - sparse_categorical_accuracy: 0.6846


In [54]:
gemma_lm.backbone.save_lora_weights("test.lora.h5")

In [61]:
template = "Instruction:\n{instruction}\n\nInput:{input_str}\n\nResponse:\n{response}"
instruction = "You are a Quant Agent that generate sentiments for financial news headlines from the perspective of a retail investor. Your response should be either \"neutral\", \"negative\", or \"positive\", and nothing else."
input_str = "McDonald's sales fall globally for first time in more than three years"

prompt = template.format(
    instruction = instruction,
    input_str = input_str,
    response = ""
)


print("Start generating answer...")
output = gemma_lm.generate(prompt, max_length = len(prompt) + 10)
print(output)

Start generating answer...
Instruction:
You are a Quant Agent that generate sentiments for financial news headlines from the perspective of a retail investor. Your response should be either "neutral", "negative", or "positive", and nothing else.

Input:McDonald's sales fall globally for first time in more than three years

Response:
The first year 2.

The first 20. 2.0.0)
Your response to the second.1000.

The first-year 1.0.e. 10.00.00000 1000.000000 million of 201 m/ms.


In [ ]:
sentiment_dataset_combined[0]